In [ ]:
from hyrax import Hyrax
import numpy as np

# producer side Hyrax instance
p_h = Hyrax()

In [ ]:
data_request = {
    "stream_source": {
        "data": {
            "dataset_class": "HyraxRandomDataset",
            "data_location": "./",
            "primary_id_field": "object_id",
            "fields": ["image"],
        }
    }
}
p_h.set_config("data_request", data_request, over_write=True)

In [ ]:
# Create a data provider that we can use in a generator to produce data for inference
ds = p_h.prepare()


# create a generator that will produce outputs from Hyrax's RandomDataset
def get_infer_data(n):
    for i in range(n):
        index = i % len(ds["stream_source"])
        data = ds["stream_source"][index]
        yield data

In [ ]:
# test the generator
for x in get_infer_data(2):
    print(x)

The following would likely be the same dataset class that was used to train the model since it would include the required collation functions.
Here it's just a minimal example that shows that we _could_ use a custom, field-level collation function.

In [ ]:
from hyrax.datasets import HyraxDataset


class MyDatasetClass(HyraxDataset):
    def __init__(self, config, data_location):
        super().__init__(config)

    def get_image(self, index):
        return None

    def collate_image(self, batch):
        # do the collation here
        # return e.g. collated_batch _and_ mask
        # stack all the images in the batch into a single numpy array
        images = []
        for i in range(len(batch)):
            images.append(batch[i]["data"]["image"])
        batch = {
            "data": {
                "image": np.array(images),
            },
        }
        return batch

    # def collate(self, batch):
    #     pass

    def __len__(self):
        # return the number of items in the dataset
        return 0

In [ ]:
# consumer side Hyrax instance
c_h = Hyrax()
c_h.set_config("model.name", "HyraxLoopback")
c_h.set_config("infer_stream.model_weights_file", "foo.bar")

In [ ]:
# define a minimal data request that uses the custom dataset class
# we want a simple way to provide a custom collate_light_curve method
data_request = {
    "infer": {
        "data": {
            "dataset_class": "MyDatasetClass",
            "data_location": "./",
            "primary_id_field": "object_id",
        }
    }
}

c_h.set_config("data_request", data_request, over_write=True)

In [ ]:
dp = c_h.prepare()

In [ ]:
# Get a sample from the data source defined at the top of the notebook
sample = ds["stream_source"].sample_data()
print(sample)
# Use the collation function from a different data provider
col_sample = dp["infer"].collate([sample])

with c_h.infer_stream(sample_batch=sample) as session:
    # Process the sample we already pulled, then keep going
    session.process(sample)
    batch = []
    for x in get_infer_data(20):
        # create batches of 5.
        batch.append(x)
        if len(batch) == 5:
            data = dp["infer"].collate(batch)
            results = session.process(data)
            print(
                f"Processed {len(batch)} items, ids {[x['object_id'] for x in batch]}. Output shape {results.shape}"
            )
            print(f"Diff: {np.sum(np.abs(results - data['data']['image']))}")
            batch = []